In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("oil_stock.silver.market_daily")

In [0]:
w = Window.partitionBy("symbol").orderBy("trading_date")

returns_df = (
    silver_df
    .withColumn(
        'previous_close',
        F.lag('close').over(w)
    )
    .withColumn(
        'daily_return',
        (F.col('close') / F.col('previous_close')) - 1
    )
)
display(returns_df)

In [0]:
gold_daily_df = (
    returns_df
    .groupBy("trading_date")
    .agg(
        F.max(
            F.when(F.col("symbol") == "EQNR.OL", F.col("close"))
        ).alias("eqnr_price"),

        F.max(
            F.when(F.col("symbol") == "BZ=F", F.col("close"))
        ).alias("brent_price"),

        F.max(
            F.when(F.col("symbol") == "EQNR.OL", F.col("daily_return"))
        ).alias("eqnr_daily_return"),

        F.max(
            F.when(F.col("symbol") == "BZ=F", F.col("daily_return"))
        ).alias("brent_daily_return")
    )
    .filter(
        F.col("eqnr_price").isNotNull() &
        F.col("brent_price").isNotNull()
    )
    .orderBy("trading_date")
)

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.market_daily")